# Benchmark Metrics Visualization (12 Model Configurations)

Generates comparative and per-metric figures matching Table 7 and Figure 2-4 in the manuscript:
- Single-Turn vs. Multi-Turn vs. Macro Strict AST comparison (Figure 2)
- Five-metric radar profile for representative models (Figure 3)
- On-premise Accuracy vs. Peak VRAM trade-off (Figure 4)
- Standalone horizontal bar charts per metric

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya visual akademik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.labelweight'] = 'bold'

OUTPUT_DIR = "Grafik_Hasil_10_Pilar_Fix"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[*] Folder penyimpanan grafik: {OUTPUT_DIR}")

In [ ]:
# Standardized 12-model evaluation metrics matching Table 7 in the manuscript.
data = [
    {"Category": "Proprietary Cloud", "Model": "Claude-Opus-5", "Params": "Cloud", "ST_ToolAcc": 89.09, "ST_StrictAST": 87.27, "MT_StrictAST": 64.42, "Overall_StrictAST": 75.85, "ArgMatch": 87.27, "MacroF1": 78.40, "FormatValid": 99.01, "Latency": 3.94, "Throughput": 28.40, "BLEU4": 9.40, "ROUGEL": 42.01, "VRAM": 0.0, "VRAM_Label": "Cloud API"},
    {"Category": "Proprietary Cloud", "Model": "MiniMax-M3", "Params": "MoE", "ST_ToolAcc": 87.27, "ST_StrictAST": 85.45, "MT_StrictAST": 21.88, "Overall_StrictAST": 53.67, "ArgMatch": 85.45, "MacroF1": 61.20, "FormatValid": 97.03, "Latency": 5.43, "Throughput": 18.20, "BLEU4": 10.19, "ROUGEL": 43.52, "VRAM": 0.0, "VRAM_Label": "Cloud API"},
    {"Category": "Mid-to-Large Open", "Model": "Gemma4-31B", "Params": "31B", "ST_ToolAcc": 94.55, "ST_StrictAST": 85.45, "MT_StrictAST": 64.17, "Overall_StrictAST": 74.81, "ArgMatch": 85.45, "MacroF1": 76.12, "FormatValid": 100.00, "Latency": 1.35, "Throughput": 42.50, "BLEU4": 3.11, "ROUGEL": 30.75, "VRAM": 28.0, "VRAM_Label": ">28 GB"},
    {"Category": "Mid-to-Large Open", "Model": "GPT-OSS-20B", "Params": "20B", "ST_ToolAcc": 89.09, "ST_StrictAST": 87.27, "MT_StrictAST": 57.10, "Overall_StrictAST": 72.19, "ArgMatch": 87.27, "MacroF1": 74.90, "FormatValid": 97.44, "Latency": 2.65, "Throughput": 22.80, "BLEU4": 1.84, "ROUGEL": 28.85, "VRAM": 18.0, "VRAM_Label": "~18 GB"},
    {"Category": "Mid-to-Large Open", "Model": "GPT-OSS-120B", "Params": "120B", "ST_ToolAcc": 85.45, "ST_StrictAST": 80.00, "MT_StrictAST": 59.82, "Overall_StrictAST": 69.91, "ArgMatch": 80.00, "MacroF1": 72.35, "FormatValid": 98.72, "Latency": 1.52, "Throughput": 38.20, "BLEU4": 1.74, "ROUGEL": 28.11, "VRAM": 75.0, "VRAM_Label": ">75 GB"},
    {"Category": "Mid-to-Large Open", "Model": "Nemotron-30B", "Params": "30B", "ST_ToolAcc": 61.82, "ST_StrictAST": 61.82, "MT_StrictAST": 14.24, "Overall_StrictAST": 38.03, "ArgMatch": 61.82, "MacroF1": 34.50, "FormatValid": 68.59, "Latency": 3.58, "Throughput": 18.40, "BLEU4": 0.86, "ROUGEL": 14.52, "VRAM": 26.0, "VRAM_Label": ">26 GB"},
    {"Category": "Proposed LoRA SLM", "Model": "TelkomNusa-LLM-7B", "Params": "7.61B", "ST_ToolAcc": 89.09, "ST_StrictAST": 83.64, "MT_StrictAST": 33.04, "Overall_StrictAST": 58.34, "ArgMatch": 83.64, "MacroF1": 74.20, "FormatValid": 100.00, "Latency": 11.02, "Throughput": 4.85, "BLEU4": 0.00, "ROUGEL": 30.24, "VRAM": 9.2, "VRAM_Label": "~9.2 GB (L4)"},
    {"Category": "Proposed LoRA SLM", "Model": "TelkomNusa-SLM-3B", "Params": "3.09B", "ST_ToolAcc": 85.45, "ST_StrictAST": 72.73, "MT_StrictAST": 16.38, "Overall_StrictAST": 44.55, "ArgMatch": 72.73, "MacroF1": 68.45, "FormatValid": 100.00, "Latency": 10.30, "Throughput": 5.12, "BLEU4": 0.00, "ROUGEL": 29.77, "VRAM": 5.8, "VRAM_Label": "~5.8 GB (T4)"},
    {"Category": "Open Baseline", "Model": "Llama-3.2-3B-Instruct", "Params": "3.21B", "ST_ToolAcc": 81.82, "ST_StrictAST": 78.18, "MT_StrictAST": 3.99, "Overall_StrictAST": 41.09, "ArgMatch": 78.18, "MacroF1": 42.10, "FormatValid": 100.00, "Latency": 22.03, "Throughput": 2.15, "BLEU4": 8.70, "ROUGEL": 44.47, "VRAM": 6.1, "VRAM_Label": "~6.1 GB (T4)"},
    {"Category": "Open Baseline", "Model": "Qwen2.5-7B-Instruct", "Params": "7.61B", "ST_ToolAcc": 85.45, "ST_StrictAST": 81.82, "MT_StrictAST": 0.00, "Overall_StrictAST": 40.91, "ArgMatch": 81.82, "MacroF1": 41.80, "FormatValid": 98.18, "Latency": 5.93, "Throughput": 8.58, "BLEU4": 74.20, "ROUGEL": 79.50, "VRAM": 9.2, "VRAM_Label": "~9.2 GB (L4)"},
    {"Category": "Open Baseline", "Model": "Qwen2.5-3B-Instruct", "Params": "3.09B", "ST_ToolAcc": 67.27, "ST_StrictAST": 61.82, "MT_StrictAST": 4.13, "Overall_StrictAST": 32.98, "ArgMatch": 61.82, "MacroF1": 47.26, "FormatValid": 99.01, "Latency": 7.65, "Throughput": 6.63, "BLEU4": 5.78, "ROUGEL": 38.86, "VRAM": 5.8, "VRAM_Label": "~5.8 GB (T4)"},
    {"Category": "Open Baseline", "Model": "Phi-4-mini", "Params": "3.8B", "ST_ToolAcc": 10.91, "ST_StrictAST": 10.91, "MT_StrictAST": 0.00, "Overall_StrictAST": 5.46, "ArgMatch": 10.91, "MacroF1": 4.80, "FormatValid": 97.03, "Latency": 6.68, "Throughput": 7.40, "BLEU4": 3.68, "ROUGEL": 22.32, "VRAM": 6.8, "VRAM_Label": "~6.8 GB (T4)"}
]
df = pd.DataFrame(data)

## Bagian I: Visualisasi Gabungan (Multi-Metric Visualizations)
Grafik gabungan dirancang untuk memadatkan analisis komparatif melintasi pilar-pilar penting ke dalam 1 visual beresolusi tinggi.

In [ ]:
# Grafik Gabungan 1: Grouped Bar Chart (Single-Turn vs Multi-Turn vs Overall Strict AST)
fig, ax = plt.subplots(figsize=(13, 6), dpi=300)
x = np.arange(len(df))
w = 0.26

r1 = ax.bar(x - w, df["ST_StrictAST"], w, label="Single-Turn Strict AST (%)", color="#1f77b4", edgecolor="black", linewidth=0.6)
r2 = ax.bar(x, df["MT_StrictAST"], w, label="Multi-Turn Strict AST (%)", color="#ff7f0e", edgecolor="black", linewidth=0.6)
r3 = ax.bar(x + w, df["Overall_StrictAST"], w, label="Overall Strict AST (%)", color="#2ca02c", edgecolor="black", linewidth=0.6)

ax.set_ylabel("Akurasi Eksekusi (%)")
ax.set_title("Komparasi Retensi Nalar BSS: Single-Turn vs. Multi-Turn vs. Overall AST", pad=12)
ax.set_xticks(x)
ax.set_xticklabels(df["Model"], rotation=30, ha="right", fontsize=9, fontweight="medium")
ax.set_ylim(0, 105)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Grafik_Gabungan_1_ST_vs_MT_Strict_AST.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Grafik Gabungan 2: Radar Chart 5 Dimensi Eksekusi BSS
radar_metrics = ["Overall_StrictAST", "ST_ToolAcc", "ArgMatch", "MacroF1", "FormatValid"]
radar_labels = ["Strict AST", "Tool Selection", "Arg Match", "Macro F1", "Format Validity"]
N_angles = len(radar_labels)
angles = np.linspace(0, 2 * np.pi, N_angles, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), dpi=300)
key_models = ["TelkomNusa-SLM-3B", "Gemma4-31B", "GPT-OSS-120B", "Llama-3.2-3B-Instruct", "Qwen2.5-3B-Instruct"]
radar_colors = ["#d62728", "#1f77b4", "#9467bd", "#2ca02c", "#7f7f7f"]

for km, c in zip(key_models, radar_colors):
    sub = df[df["Model"] == km].iloc[0]
    vals = [sub[m] for m in radar_metrics]
    vals += vals[:1]
    is_our = "TelkomNusa" in km
    ax.plot(angles, vals, color=c, linewidth=2.5 if is_our else 1.5, label=f"{km} ({sub['Params']})")
    ax.fill(angles, vals, color=c, alpha=0.2 if is_our else 0.04)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontweight="bold", fontsize=10)
ax.set_ylim(0, 105)
ax.set_title("Profil Kesiapan Operasional BSS Melintasi Core Metrics Eksekusi", pad=35, y=1.05, fontsize=12, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.05), fontsize=8.5, frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Grafik_Gabungan_2_Radar_5_Pilar_Eksekusi.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Performance-VRAM trade-off across on-premise models.
def get_model_color(name):
    if "TelkomNusa-SLM-3B" in name: return "#d62728"
    elif "TelkomNusa" in name: return "#ff7f0e"
    elif "Gemma" in name or "GPT-OSS" in name or "Nemotron" in name: return "#1f77b4"
    elif "Claude" in name or "MiniMax" in name: return "#5c6bc0"
    else: return "#7f7f7f"

on_prem = df[df['VRAM'] > 0].copy()
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
for _, row in on_prem.iterrows():
    is_our = 'TelkomNusa' in row['Model']
    ax.scatter(row['VRAM'], row['Overall_StrictAST'], s=220 if is_our else 110, color=get_model_color(row['Model']), edgecolor='black', linewidth=0.9, zorder=4 if is_our else 3)
    ax.annotate(row['Model'], xy=(row['VRAM'], row['Overall_StrictAST']), xytext=(row['VRAM'] + 1.0, row['Overall_StrictAST'] + 1.5), fontsize=8.5, fontweight='bold' if is_our else 'normal')
ax.axvline(x=16.0, color='#666666', linestyle=':', linewidth=1.2, label='16 GB Single-GPU Limit')
ax.set_xlabel('Peak VRAM Consumption (GB)')
ax.set_ylabel('Macro Strict AST Accuracy (%)')
ax.set_title('On-Premise Accuracy vs. VRAM Trade-Off', pad=12)
ax.set_xlim(0, 82)
ax.set_ylim(0, 86)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Grafik_Gabungan_3_VRAM_vs_Accuracy.png"), bbox_inches="tight")
plt.show()

## Bagian II: Visualisasi Individual (Standalone Per Pilar Metrik)
Grafik individual dapat disematkan satu per satu ke dalam sub-bab analisis skripsi yang bersangkutan.

In [ ]:
# [Pilar 1] Strict AST Accuracy (Sorted Bar Horizontal)
df_sorted_ast = df.sort_values("Overall_StrictAST", ascending=True)
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.barh(df_sorted_ast["Model"], df_sorted_ast["Overall_StrictAST"], 
                color=[get_model_color(m) for m in df_sorted_ast["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Overall Strict AST Accuracy (%)")
plt.title("[Pilar 1] Akurasi Pemanggilan Fungsi Deterministik (Strict Full AST Match)", pad=10)
plt.xlim(0, 100)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height()/2, f"{w:.2f}%", va="center", fontsize=8.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_1_Strict_AST_Accuracy.png"), bbox_inches="tight")
plt.show()

In [ ]:
# [Pilar 3] Argument Match Rate
df_sorted_arg = df.sort_values("ArgMatch", ascending=True)
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.barh(df_sorted_arg["Model"], df_sorted_arg["ArgMatch"], 
                color=[get_model_color(m) for m in df_sorted_arg["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Argument Match Rate (%)")
plt.title("[Pilar 3] Ketepatan Ekstraksi Pasangan Kunci-Nilai Argumen (Argument Match)", pad=10)
plt.xlim(0, 105)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height()/2, f"{w:.2f}%", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_3_Argument_Match_Rate.png"), bbox_inches="tight")
plt.show()

In [ ]:
# [Pilar 4] Macro F1-Score
df_sorted_f1 = df.sort_values("MacroF1", ascending=True)
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.barh(df_sorted_f1["Model"], df_sorted_f1["MacroF1"], 
                color=[get_model_color(m) for m in df_sorted_f1["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Macro F1-Score (%)")
plt.title("[Pilar 4] Keseimbangan Distribusi Prediksi Melintasi 23 API BSS (Macro F1)", pad=10)
plt.xlim(0, 100)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height()/2, f"{w:.2f}%", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_4_Macro_F1_Score.png"), bbox_inches="tight")
plt.show()

In [ ]:
# [Pilar 8 & 9] Latensi vs Throughput (Dual-Axis / Efficiency)
df_sorted_lat = df.sort_values("Latency", ascending=False)
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.barh(df_sorted_lat["Model"], df_sorted_lat["Latency"], 
                color=[get_model_color(m) for m in df_sorted_lat["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Rata-Rata Latensi Inferensi (detik/soal) -> Makin Pendek Makin Cepat")
plt.title("[Pilar 8] Waktu Respon Inferensi Kueri Pelanggan (Average Latency)", pad=10)
plt.xlim(0, 26)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.4, bar.get_y() + bar.get_height()/2, f"{w:.2f}s", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_8_Average_Latency.png"), bbox_inches="tight")
plt.show()

# [Pilar 9] Throughput
df_sorted_tp = df.sort_values("Throughput", ascending=True)
plt.figure(figsize=(10, 6), dpi=300)
bars = plt.barh(df_sorted_tp["Model"], df_sorted_tp["Throughput"], 
                color=[get_model_color(m) for m in df_sorted_tp["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Throughput Generasi (token/detik) -> Makin Panjang Makin Cepat")
plt.title("[Pilar 9] Kecepatan Generasi Token Output (Generation Throughput)", pad=10)
plt.xlim(0, 52)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.8, bar.get_y() + bar.get_height()/2, f"{w:.2f} tok/s", va="center", fontsize=8.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_9_Generation_Throughput.png"), bbox_inches="tight")
plt.show()

In [ ]:
# [Pilar 10] VRAM Memory Footprint
df_local = df[df["VRAM"] > 0].sort_values("VRAM", ascending=True)
plt.figure(figsize=(9, 5), dpi=300)
bars = plt.barh(df_local["Model"], df_local["VRAM"], 
                color=[get_model_color(m) for m in df_local["Model"]],
                edgecolor="black", linewidth=0.7)
plt.xlabel("Estimasi Konsumsi Memori GPU Inferensi (Gigabytes VRAM)")
plt.title("[Pilar 10] Jejak Alokasi Memori Kartu Grafis (Peak VRAM Footprint)", pad=10)
plt.xlim(0, 85)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height()/2, f"{w:.1f} GB", va="center", fontsize=8.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Pilar_10_VRAM_Memory_Footprint.png"), bbox_inches="tight")
plt.show()